# Geração da amostra via API pública da Steam

Notebook auxiliar, **fora da entrega**. Ele monta o arquivo `steam_reviews_amostra_2026.csv` que o notebook principal lê.

**Desenho amostral**

1. Pool de candidatos: jogos mais populares da Steam segundo o SteamSpy (2.000 jogos ordenados por número de donos).
2. Pré-filtro barato: total de reviews (todas as línguas) entre `MIN_REVIEWS_POOL` e `MAX_REVIEWS_POOL`. Isso tira jogos pequenos demais e os gigantes (CS2, Elden Ring, etc.), cujos reviews de 2026 sozinhos passam de 100 mil e tornariam a coleta inviável.
3. Sorteio com seed fixa. Para cada jogo sorteado, a API confirma o total de reviews em inglês; se ficar fora de `[MIN_EN, MAX_EN]` o jogo é descartado e o próximo da fila entra.
4. Coleta: `filter=recent` devolve do mais novo para o mais antigo. Reviews criados a partir de `DATA_CORTE` são pulados; os `N_POR_JOGO` seguintes são guardados.
5. O corte em 2026-01-01 garante janela mínima de ~8 meses entre o review e a coleta (set/2026), tempo para observar se o jogador voltou a jogar.

**Viés assumido:** dentro de cada jogo a amostra são os 5.000 reviews mais recentes antes do corte, não um sorteio aleatório. Isso será declarado no notebook principal.

Cada jogo aceito é salvo em `raw/<appid>.json`, então o notebook pode ser interrompido e retomado sem refazer downloads.

In [6]:
import requests
import pandas as pd
import numpy as np
import json
import time
import datetime as dt
from pathlib import Path

## Parâmetros

In [7]:
SEED = 42
N_JOGOS = 30              # jogos na amostra final
N_POR_JOGO = 5000         # reviews por jogo -> 150.000 no total
DATA_CORTE = dt.datetime(2026, 1, 1)   # só reviews criados ANTES desta data
DATA_COLETA = dt.date.today()

# pré-filtro no pool (todas as línguas, número do SteamSpy)
MIN_REVIEWS_POOL = 15_000
MAX_REVIEWS_POOL = 120_000

# checagem real na API (reviews em inglês)
MIN_EN = 10_000
MAX_EN = 60_000

MAX_PAGINAS_PULAR = 120   # páginas de 2026 que aceitamos pular antes de desistir do jogo
PAUSA = 0.5               # segundos entre requisições à Steam
PAGINAS_STEAMSPY = [0, 1] # 1.000 jogos por página; SteamSpy limita a 1 req/min neste endpoint

PASTA_SAIDA = Path('.')
PASTA_RAW = PASTA_SAIDA / 'raw'
PASTA_RAW.mkdir(exist_ok=True)

CORTE_TS = int(DATA_CORTE.timestamp())

## 1. Pool de candidatos (SteamSpy)

In [8]:
# baixa as páginas do ranking por número de donos
paginas = []
for p in PAGINAS_STEAMSPY:
    r = requests.get('https://steamspy.com/api.php', params={'request': 'all', 'page': p}, timeout=60)
    paginas.append(pd.DataFrame(r.json()).T)
    if p != PAGINAS_STEAMSPY[-1]:
        time.sleep(61)   # limite do SteamSpy para request=all

pool = pd.concat(paginas, ignore_index=True)
pool['appid'] = pool['appid'].astype(int)
pool['total_reviews'] = pool['positive'].astype(int) + pool['negative'].astype(int)

# pré-filtro: nem pequeno demais, nem gigante
pool = pool[pool['total_reviews'].between(MIN_REVIEWS_POOL, MAX_REVIEWS_POOL)]
pool = pool[['appid', 'name', 'developer', 'positive', 'negative', 'total_reviews', 'owners']].reset_index(drop=True)
pool.to_csv(PASTA_SAIDA / 'pool_candidatos.csv', index=False)

print(f'{len(pool)} jogos candidatos')
pool.head()

1008 jogos candidatos


,appid,name,developer,positive,negative,total_reviews,owners
0,291480,Warface: Clutch,MY.GAMES,54554,27643,82197,"20,000,000 .. 50,000,000"
1,899770,Last Epoch,Eleventh Hour Games,88027,22596,110623,"20,000,000 .. 50,000,000"
2,386360,SMITE,Titan Forge Games,94368,25428,119796,"10,000,000 .. 20,000,000"
3,755790,Ring of Elysium,Aurora Studio,74345,25492,99837,"10,000,000 .. 20,000,000"
4,550650,Black Squad,"VALOFE, NS STUDIO",62042,19672,81714,"10,000,000 .. 20,000,000"


## 2. Coleta

A função pagina os reviews de um jogo do mais novo para o mais antigo, pula os criados a partir de `DATA_CORTE` e para ao juntar `N_POR_JOGO`. Devolve `None` (com o motivo) quando o jogo não serve.

In [9]:
def pegar_pagina(url, params):
    # até 5 tentativas com espera crescente: 5, 10, 20, 40, 80 s
    for tentativa in range(5):
        try:
            return requests.get(url, params=params, timeout=60).json()
        except (requests.exceptions.RequestException, ValueError) as e:
            espera = 5 * 2 ** tentativa
            print(f'         erro de rede ({type(e).__name__}), nova tentativa em {espera}s')
            time.sleep(espera)
    raise RuntimeError('Steam não respondeu após 5 tentativas')


def baixar_reviews(appid):
    url = f'https://store.steampowered.com/appreviews/{appid}'
    params = {'json': 1, 'filter': 'recent', 'language': 'english',
              'num_per_page': 100, 'cursor': '*', 'purchase_type': 'all'}
    reviews, vistos = [], set()
    paginas_puladas = 0

    while len(reviews) < N_POR_JOGO:
        r = pegar_pagina(url, params)
        time.sleep(PAUSA)

        if r.get('success') != 1 or not r.get('reviews'):
            break   # acabou o histórico do jogo

        # checagem de tamanho só na primeira página
        if params['cursor'] == '*':
            total_en = r['query_summary']['total_reviews']
            if not (MIN_EN <= total_en <= MAX_EN):
                return None, f'{total_en} reviews em inglês fora de [{MIN_EN}, {MAX_EN}]'

        for rev in r['reviews']:
            if rev['recommendationid'] in vistos:
                continue
            vistos.add(rev['recommendationid'])
            if rev['timestamp_created'] < CORTE_TS:
                reviews.append(rev)

        if not reviews:
            paginas_puladas += 1
            if paginas_puladas >= MAX_PAGINAS_PULAR:
                return None, f'mais de {MAX_PAGINAS_PULAR} páginas só com reviews de 2026'

        params['cursor'] = r['cursor']

    if len(reviews) < N_POR_JOGO:
        return None, f'só {len(reviews)} reviews antes do corte'

    return reviews[:N_POR_JOGO], None

In [10]:
# sorteia a ordem dos candidatos e vai aceitando até completar N_JOGOS
rng = np.random.default_rng(SEED)
ordem = rng.permutation(len(pool))

aceitos, descartados = [], []
inicio = time.time()

for i in ordem:
    if len(aceitos) == N_JOGOS:
        break
    appid, nome = int(pool.loc[i, 'appid']), pool.loc[i, 'name']
    arquivo = PASTA_RAW / f'{appid}.json'

    if arquivo.exists():   # já baixado numa execução anterior
        aceitos.append((appid, nome))
        print(f'[{len(aceitos):2d}/{N_JOGOS}] {nome} (cache)')
        continue

    reviews, motivo = baixar_reviews(appid)
    if reviews is None:
        descartados.append((appid, nome, motivo))
        print(f'       descartado: {nome} -> {motivo}')
        continue

    json.dump(reviews, open(arquivo, 'w'))
    aceitos.append((appid, nome))
    print(f'[{len(aceitos):2d}/{N_JOGOS}] {nome}  ({(time.time() - inicio) / 60:.1f} min)')

pd.DataFrame(aceitos, columns=['appid', 'game']).to_csv(PASTA_SAIDA / 'jogos_sorteados_2026.csv', index=False)
pd.DataFrame(descartados, columns=['appid', 'game', 'motivo']).to_csv(PASTA_SAIDA / 'jogos_descartados_2026.csv', index=False)
print(f'\n{len(aceitos)} jogos aceitos, {len(descartados)} descartados, {(time.time() - inicio) / 60:.1f} min')

       descartado: Serious Sam 4 -> 5900 reviews em inglês fora de [10000, 60000]
[ 1/30] Neverwinter (cache)
       descartado: Half-Life: Alyx -> 70202 reviews em inglês fora de [10000, 60000]
[ 2/30] Sleeping Dogs: Definitive Edition (cache)
       descartado: Football Manager 2021 -> 9485 reviews em inglês fora de [10000, 60000]
       descartado: SMITE -> 67960 reviews em inglês fora de [10000, 60000]
[ 3/30] Call of Duty: World at War (cache)
[ 4/30] Stoneshard (cache)
[ 5/30] Microsoft Flight Simulator X: Steam Edition (cache)
[ 6/30] Little Nightmares II (cache)
       descartado: Sniper Elite 3 -> 9738 reviews em inglês fora de [10000, 60000]
[ 7/30] Abiotic Factor (cache)
[ 8/30] Dorfromantik (cache)
[ 9/30] Yakuza Kiwami (cache)
       descartado: Contagion -> 8643 reviews em inglês fora de [10000, 60000]
[10/30] The Binding of Isaac (cache)
[11/30] Thief Simulator (cache)
       descartado: We Were Here Forever -> 6733 reviews em inglês fora de [10000, 60000]
[12/30] Days G

## 3. Montar o CSV

Colunas com os mesmos nomes da base do Kaggle, para o notebook principal mudar o mínimo. Novidades da API atual: `refunded`, `primarily_steam_deck` e `app_release_date`.

In [11]:
partes = []
for appid, nome in aceitos:
    reviews = json.load(open(PASTA_RAW / f'{appid}.json'))
    parte = pd.json_normalize(reviews)
    parte.insert(1, 'game', nome)
    parte.insert(0, 'appid', appid)
    partes.append(parte)

df = pd.concat(partes, ignore_index=True)

# author.steamid -> author_steamid etc.
df.columns = df.columns.str.replace('.', '_', regex=False)

# campos de perfil sem uso analítico
df = df.drop(columns=['author_personaname', 'author_persona_status', 'author_profile_url',
                      'author_avatar', 'reactions'])

# booleanos -> 0/1 como na base do Kaggle
for col in ['voted_up', 'steam_purchase', 'received_for_free', 'written_during_early_access',
            'refunded', 'primarily_steam_deck']:
    df[col] = df[col].astype(int)

df['weighted_vote_score'] = df['weighted_vote_score'].astype(float)
df['app_release_date'] = pd.to_numeric(df['app_release_date'], errors='coerce')

ordem_colunas = ['recommendationid', 'appid', 'game', 'author_steamid', 'author_num_games_owned',
                 'author_num_reviews', 'author_playtime_forever', 'author_playtime_last_two_weeks',
                 'author_playtime_at_review', 'author_last_played', 'language', 'review',
                 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny',
                 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free',
                 'written_during_early_access', 'refunded', 'primarily_steam_deck', 'app_release_date']
df = df[ordem_colunas]
df.shape

(150000, 25)

## 4. Sanidade e gravação

In [12]:
print('reviews por jogo:')
print(df['game'].value_counts().to_string())
print()
print('período dos reviews:',
      pd.to_datetime(df['timestamp_created'].min(), unit='s').date(), '->',
      pd.to_datetime(df['timestamp_created'].max(), unit='s').date())
print('duplicados:', df['recommendationid'].duplicated().sum())
print('reembolsados:', df['refunded'].sum())

reviews por jogo:
game
Neverwinter                                                5000
Sleeping Dogs: Definitive Edition                          5000
Call of Duty: World at War                                 5000
Stoneshard                                                 5000
Microsoft Flight Simulator X: Steam Edition                5000
Little Nightmares II                                       5000
Abiotic Factor                                             5000
Dorfromantik                                               5000
Yakuza Kiwami                                              5000
The Binding of Isaac                                       5000
Thief Simulator                                            5000
Days Gone                                                  5000
AdVenture Capitalist                                       5000
Dwarf Fortress                                             5000
Assassin's Creed Unity                                     5000
Red Orchestra 2: 

In [13]:
df.to_csv(PASTA_SAIDA / 'steam_reviews_amostra_2026.csv', index=False)

meta = {'data_coleta': str(DATA_COLETA), 'data_corte': str(DATA_CORTE.date()), 'seed': SEED,
        'n_jogos': len(aceitos), 'n_reviews': len(df), 'min_en': MIN_EN, 'max_en': MAX_EN}
json.dump(meta, open(PASTA_SAIDA / 'amostra_2026_meta.json', 'w'), indent=2)
meta

{'data_coleta': '2026-09-14',
 'data_corte': '2026-01-01',
 'seed': 42,
 'n_jogos': 30,
 'n_reviews': 150000,
 'min_en': 10000,
 'max_en': 60000}